# Paper 3 — Magnetic Tunnel Junction Enabled All-Spin Stochastic Spiking Neural Network

**G. Srinivasan, A. Sengupta, and K. Roy**, DATE 2017, pp. 530–535. DOI: [10.23919/DATE.2017.7927045](https://doi.org/10.23919/DATE.2017.7927045).

This is a reduced-order computational reconstruction, not a bit-for-bit reproduction. All default device parameters are illustrative, not claimed to be extracted from the paper. The notebook builds a reproducible device → synapse → crossbar → stochastic-neuron → toy-task workflow.

## 1. Equations

For binary MTJ state \(m\in\{-1,+1\}\),
\[
G(m)=G_{AP}+\frac{m+1}{2}(G_P-G_{AP}).
\]
Thermally activated compact switching:
\[
\Gamma(I)=f_0e^{-\Delta_{\rm eff}(I)},\quad
P_{\rm sw}=1-e^{-\Gamma(I)t_p}.
\]
Timing-dependent update envelope:
\[
P_{\rm upd}(\Delta t)=p_{\max}e^{-|\Delta t|/\tau},\quad
\Delta t=t_{\rm post}-t_{\rm pre}.
\]
The sign convention below treats post-after-pre as potentiation. Verify the actual pulse convention against the source paper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
rng = np.random.default_rng(17)
plt.rcParams['figure.figsize'] = (7,4)
plt.rcParams['axes.grid'] = True

## 2. MTJ compact model and switching curves

In [ ]:
@dataclass
class MTJ:
    Gp: float = 20e-6
    Gap: float = 5e-6
    f0: float = 1e9
    Delta0: float = 20.
    xi: float = .8
    I0: float = 50e-6
    Delta_min: float = .5
    def conductance(self,m):
        return self.Gap + .5*(np.asarray(m)+1)*(self.Gp-self.Gap)
    def barrier(self,I):
        return np.maximum(self.Delta_min,self.Delta0*(1-self.xi*np.abs(np.asarray(I))/self.I0))
    def rate(self,I):
        return self.f0*np.exp(-self.barrier(I))
    def p_switch(self,I,tp):
        return -np.expm1(-self.rate(I)*np.asarray(tp))
    def step(self,m,I,tp,generator=rng):
        return -int(m) if generator.random()<float(self.p_switch(I,tp)) else int(m)

mtj=MTJ()
tp=np.logspace(-10,-6,250)
for I in [10e-6,25e-6,40e-6,50e-6]:
    plt.semilogx(tp*1e9,mtj.p_switch(I,tp),label=f'{I*1e6:.0f} µA')
plt.xlabel('Pulse width (ns)'); plt.ylabel('Switching probability')
plt.title('Illustrative MTJ switching'); plt.legend(); plt.show()

In [ ]:
# Monte Carlo check of the analytic switching probability
I0,tp0,N=40e-6,20e-9,100_000
pth=float(mtj.p_switch(I0,tp0))
pmc=(rng.random(N)<pth).mean()
print(f'Analytic={pth:.5f}; Monte Carlo={pmc:.5f}; N={N:,}')

## 3. Stochastic binary synapse and timing rule

In [ ]:
@dataclass
class TimingSynapse:
    tau: float = 20e-3
    pmax: float = .35
    def p_update(self,dt):
        return self.pmax*np.exp(-np.abs(dt)/self.tau)
    def update(self,w,dt,generator=rng):
        if generator.random()<float(self.p_update(dt)):
            w=min(1,w+1) if dt>=0 else max(-1,w-1)
        return int(w)

syn=TimingSynapse()
dts=np.linspace(-80e-3,80e-3,401)
plt.plot(dts*1e3,syn.p_update(dts))
plt.xlabel('Delta t = t_post - t_pre (ms)'); plt.ylabel('Update probability')
plt.title('Timing-dependent update envelope'); plt.show()

def pairings(seq,seed=3):
    local=np.random.default_rng(seed); w=0; hist=[w]
    for dt in seq:
        w=syn.update(w,dt,local); hist.append(w)
    return np.array(hist)
seq=np.r_[np.full(100,8e-3),np.full(100,-8e-3)]
plt.plot(pairings(seq)); plt.xlabel('Pairing'); plt.ylabel('Synaptic state')
plt.title('Potentiation then depression'); plt.show()

## 4. Differential crossbar

Ideal Ohmic current summation is
\[
\mathbf{i}_{out}=\mathbf{x}^{T}(G^+-G^-).
\]
This omits line resistance, sneak paths, device nonlinearity, read disturb, and peripheral circuits.

In [ ]:
def crossbar(x,Gplus,Gminus):
    return np.asarray(x) @ (np.asarray(Gplus)-np.asarray(Gminus))
nin,nout=8,4
W=rng.choice([-1,0,1],size=(nin,nout))
Gp=mtj.Gap+(W>0)*(mtj.Gp-mtj.Gap)
Gm=mtj.Gap+(W<0)*(mtj.Gp-mtj.Gap)
x=rng.uniform(0,.2,nin)
print('Differential output currents (A):',crossbar(x,Gp,Gm))

## 5. Stochastic spiking-neuron abstraction

In [ ]:
@dataclass
class StochasticNeuron:
    leak: float=.92
    threshold: float=.8
    gain: float=5.
    def run(self,drive,generator=rng):
        v=0.; volts=[]; spikes=[]; probs=[]
        for u in drive:
            v=self.leak*v+float(u)
            p=1/(1+np.exp(-self.gain*(v-self.threshold)))
            s=int(generator.random()<p)
            if s: v=0.
            volts.append(v); spikes.append(s); probs.append(p)
        return np.array(volts),np.array(spikes),np.array(probs)

drive=np.r_[np.zeros(20),np.full(30,.22),np.zeros(20),np.full(30,.22)]
v,s,p=StochasticNeuron().run(drive)
fig,ax=plt.subplots(2,1,sharex=True,figsize=(8,5))
ax[0].plot(v); ax[0].axhline(.8,ls='--',color='k'); ax[0].set_ylabel('Membrane proxy')
ax[1].stem(np.flatnonzero(s),np.ones(s.sum()),basefmt=' ')
ax[1].set_ylabel('Spikes'); ax[1].set_xlabel('Time step')
plt.tight_layout(); plt.show()
print('Spike count:',s.sum())

## 6. Synthetic temporal-pattern smoke test
This is not the paper's dataset, architecture, or benchmark accuracy.

In [ ]:
T=40
def template(c):
    z=np.zeros((T,4))
    if c==0: z[5:12,0]=1
    elif c==1: z[25:32,1]=1
    elif c==2: z[5:12,2]=1; z[25:32,2]=.6
    else: z[15:28,3]=1
    return z
def make_data(n=100,noise=.18,seed=22):
    r=np.random.default_rng(seed); X=[]; y=[]
    for c in range(4):
        for _ in range(n):
            X.append(np.clip(template(c)+noise*r.normal(size=(T,4)),0,None)); y.append(c)
    return np.array(X),np.array(y)
X,y=make_data()
def features(X):
    sums=X.sum(axis=1)
    t=np.arange(T)[None,:,None]
    cent=(X*t).sum(axis=1)/(sums+1e-9)
    return np.c_[sums,cent]
F=features(X); order=rng.permutation(len(y)); cut=int(.8*len(y))
tr,te=order[:cut],order[cut:]
mu=F[tr].mean(0); sd=F[tr].std(0)+1e-8
A=(F[tr]-mu)/sd; B=(F[te]-mu)/sd
Y=np.eye(4)[y[tr]]; Wo=np.zeros((A.shape[1],4))
for _ in range(500):
    z=A@Wo; z-=z.max(axis=1,keepdims=True)
    q=np.exp(z); q/=q.sum(axis=1,keepdims=True)
    Wo-=.15*(A.T@(q-Y)/len(tr)+1e-3*Wo)
pred=(B@Wo).argmax(axis=1)
print('Synthetic hold-out accuracy:',(pred==y[te]).mean())
cm=np.zeros((4,4),int)
for a,b in zip(y[te],pred): cm[a,b]+=1
plt.imshow(cm); plt.colorbar(label='Count')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Toy confusion matrix')
for i in range(4):
    for j in range(4): plt.text(j,i,str(cm[i,j]),ha='center',va='center')
plt.show()

## 7. First-order energy estimates

\[
E_{write}\approx I_{write}^{2}R_{write}t_p,\qquad
E_{read}\approx V_{read}^{2}G_{read}t_{read}.
\]
These are device-level approximations only; they omit drivers, interconnect, ADC/DAC, sensing, clocking, and control.

In [ ]:
def write_energy(I,R,tp): return I**2*R*tp
def read_energy(V,G,tr): return V**2*G*tr
ew=write_energy(40e-6,5e3,20e-9)
er=read_energy(.1,mtj.Gp,10e-9)
print(f'Write={ew:.3e} J; read={er:.3e} J; ratio={ew/er:.2f}')

## 8. Parameter sweeps

In [ ]:
ratios=np.linspace(.2,1,100)
for D in [10,15,20,25,30]:
    model=MTJ(Delta0=D)
    plt.plot(ratios,model.p_switch(ratios*model.I0,20e-9),label=f'Delta0={D}')
plt.xlabel('I/I0'); plt.ylabel('P(switch), 20 ns')
plt.title('Thermal-stability sensitivity'); plt.legend(); plt.show()
for tau_ms in [5,10,20,40]:
    model=TimingSynapse(tau=tau_ms*1e-3)
    plt.plot(dts*1e3,model.p_update(dts),label=f'tau={tau_ms} ms')
plt.xlabel('Delta t (ms)'); plt.ylabel('Update probability')
plt.title('Timing-window sensitivity'); plt.legend(); plt.show()

## 9. Physics-to-circuit bridge and validation checklist

| Reduced model | Higher-fidelity validation |
|---|---|
| Binary MTJ | Stochastic LLG macrospin; MuMax3/OOMMF for spatial textures |
| Phenomenological barrier | Calibrated torque-driven switching vs current, pulse width, temperature |
| Two conductances | Bias-dependent MTJ I–V, TMR, read-disturb |
| Timing update envelope | Pulse-level STDP update surface from device data |
| Ideal crossbar | ngspice/SPICE with line resistance, parasitics and nonlinear devices |
| Leaky stochastic neuron | Circuit-calibrated stochastic neuron |
| Synthetic toy task | Exact paper network, dataset and training protocol |

- [ ] Extract paper parameters and pulse conventions.
- [ ] Reproduce a single-device switching curve.
- [ ] Validate the full STDP update surface.
- [ ] Compare array currents against a circuit simulator.
- [ ] Replace the toy task with the paper's actual benchmark.
- [ ] Label every parameter as measured, paper-reported, fitted, or assumed.
- [ ] Report seeds, Monte Carlo uncertainty, and convergence tests.

## References

1. G. Srinivasan, A. Sengupta, and K. Roy, “Magnetic Tunnel Junction Enabled All-Spin Stochastic Spiking Neural Network,” DATE 2017, pp. 530–535. [DOI: 10.23919/DATE.2017.7927045](https://doi.org/10.23919/DATE.2017.7927045).
2. G. Srinivasan, A. Sengupta, and K. Roy, “Magnetic Tunnel Junction Based Long-Term Short-Term Stochastic Synapse for a Spiking Neural Network with On-Chip STDP Learning,” *Scientific Reports* 6, 29545 (2016). [DOI: 10.1038/srep29545](https://doi.org/10.1038/srep29545).
3. A. Sengupta et al., “Magnetic Tunnel Junction Mimics Stochastic Cortical Spiking Neurons,” *Scientific Reports* 6, 30039 (2016). [DOI: 10.1038/srep30039](https://doi.org/10.1038/srep30039).
4. J. Grollier et al., “Neuromorphic Spintronics,” *Nature Electronics* 3 (2020).

**Scientific limitation:** This notebook is an executable modeling scaffold. It does not claim exact reproduction of the DATE 2017 figures, measured device data, benchmark accuracy, or energy results.